# Notebook 11 — LOPO DeBERTa-v3 Level 0, chạy lại (tuần 11, mục B16)

Lượt tuần 10 (`week10_person3_full/06_lopo_deberta_colab.ipynb`) có hai vấn đề:
1. dùng **transformers 5.20.2** (T4, fp16) — bản này làm DeBERTa-v3 train hỏng (đã kiểm chứng tuần 11);
2. đặt đề bị giấu làm `eval_dataset` với `load_best_model_at_end=True` → **chọn epoch bằng chính tập kiểm tra ngoại đề**.

Lượt này: transformers 4.57.6 + bf16; train cố định 3 epoch, lấy mô hình cuối, chấm đề bị giấu đúng một lần;
dự đoán lưu kèm `row_idx`. Giữ nguyên: đầu vào L0 `type [SEP] text`, deberta-v3-base, max_length 512, lr 2e-5,
batch hiệu dụng 32, warmup 0.1, class weight balanced theo fold, seed 42 + fold, train = toàn bộ đơn vị có nhãn của 14 đề còn lại.

In [ ]:
# ===== 1. MÔI TRƯỜNG =====
# transformers 5.x làm DeBERTa-v3 train hỏng (đã kiểm chứng tuần 11) -> bắt buộc 4.x.
# Trên Colab CLI: `colab install transformers==4.57.6` rồi `colab restart-kernel` trước khi chạy notebook.
import importlib.metadata
TV = importlib.metadata.version("transformers")
assert TV.startswith("4."), f"Cần transformers 4.x, đang có {TV}"

import os, gc, json, time, random, hashlib, math, zipfile, copy
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import transformers
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score, f1_score,
                             precision_score, recall_score)
from sklearn.utils.class_weight import compute_class_weight

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
QUICK = os.environ.get("QUICK", "0") == "1"      # chạy thử: vài chục bài, 1 epoch — KHÔNG dùng số này
GPU = torch.cuda.get_device_name(0)
USE_BF16 = torch.cuda.is_bf16_supported()
print(f"transformers {TV} | torch {torch.__version__} | {GPU} | bf16={USE_BF16} | QUICK={QUICK}")
print(f"VRAM {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

In [ ]:
KEY = "LOPO_L0"

In [ ]:
# ===== 2. DỮ LIỆU + SPLIT =====
WORK = "/content"
DATA_DIR = f"{WORK}/data"
OUT_DIR = f"{WORK}/out_{KEY}" + ("_QUICK" if QUICK else "")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
TRAIN_CSV = f"{DATA_DIR}/persuade_corpus_2.0_train.csv"
TRAIN_URL = "https://drive.usercontent.google.com/download?id=13phHyDzIsb0MHyJr6q-B-qIa9P2tM135&export=download&confirm=t"
TRAIN_SHA256 = "f61319edd8bf16a982711ea0399fad59c05afaec05cdf0767f16a2c05c467e23"
LABEL2ID = {"Ineffective": 0, "Adequate": 1, "Effective": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


def log(msg):
    line = f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC] [{KEY}] {msg}"
    print(line, flush=True)
    with open(f"{OUT_DIR}/run_log.txt", "a", encoding="utf-8") as f:
        f.write(line + "\n")


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


if not os.path.exists(TRAIN_CSV):
    t0 = time.time()
    with requests.get(TRAIN_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(TRAIN_CSV + ".part", "wb") as f:
            for block in r.iter_content(chunk_size=1 << 22):
                f.write(block)
    os.replace(TRAIN_CSV + ".part", TRAIN_CSV)
    log(f"Tải train CSV xong sau {time.time() - t0:.0f}s")
assert sha256_of(TRAIN_CSV) == TRAIN_SHA256, "SHA-256 của train CSV không khớp"


def load_units():
    # Cùng quy tắc lọc với notebook 07 -> row_idx trùng với val_predictions_*.csv tuần 11
    cols = ["essay_id_comp", "full_text", "discourse_start", "discourse_end", "discourse_text",
            "discourse_type", "discourse_effectiveness", "prompt_name"]
    df = pd.read_csv(TRAIN_CSV, usecols=cols, dtype=str, low_memory=False)
    df = df.dropna(subset=["discourse_effectiveness", "discourse_text"]).copy()
    for c in ["discourse_text", "discourse_type", "essay_id_comp"]:
        df[c] = df[c].astype(str).str.strip()
    df = df[df["discourse_text"].str.len() > 0].reset_index(drop=True)
    df["row_idx"] = np.arange(len(df))
    df["prompt_name"] = df["prompt_name"].fillna("").str.replace('"', "", regex=False).str.strip()
    df["label"] = df["discourse_effectiveness"].map(LABEL2ID).astype(int)
    df["start"] = pd.to_numeric(df["discourse_start"], errors="coerce").fillna(0).astype(int)
    df["end"] = pd.to_numeric(df["discourse_end"], errors="coerce").fillna(0).astype(int)
    # Đơn vị trước/sau trong cùng bài (theo essay_id_comp, sắp theo vị trí) — như notebook 07
    s = df.sort_values(["essay_id_comp", "start", "row_idx"], kind="mergesort")
    g = s.groupby("essay_id_comp")["discourse_text"]
    for k in (1, 2):
        s[f"prev{k}"] = g.shift(k).fillna("")
        s[f"next{k}"] = g.shift(-k).fillna("")
    return s.sort_values("row_idx").reset_index(drop=True)


units = load_units()
assert len(units) == 144289, len(units)
REF = pd.read_csv(f"{WORK}/val_predictions_L1.csv")          # dự đoán L1 tuần 11 trên validation
REF_L0 = pd.read_csv(f"{WORK}/val_predictions_L0.csv")
units["split"] = np.where(units["row_idx"].isin(set(REF["row_idx"])), "val", "train")
assert units.groupby("essay_id_comp")["split"].nunique().max() == 1, "Một bài nằm ở cả train và val"
train_df = units[units["split"] == "train"].reset_index(drop=True)
val_df = units[units["split"] == "val"].reset_index(drop=True)
assert len(train_df) == 115187 and len(val_df) == 29102
assert (val_df["row_idx"].values == REF["row_idx"].values).all()
assert (val_df["label"].values == REF["true_label"].values).all()
log(f"Train {len(train_df):,} đơn vị / {train_df.essay_id_comp.nunique():,} bài | "
    f"Val {len(val_df):,} đơn vị / {val_df.essay_id_comp.nunique():,} bài (khớp split tuần 9–11)")

FULL_VAL = val_df  # giữ bản đầy đủ để so sánh
if QUICK:
    rng = np.random.RandomState(0)
    pick = lambda d, k: d[d.essay_id_comp.isin(rng.choice(d.essay_id_comp.unique(), k, replace=False))]
    train_df = pick(train_df, 80).reset_index(drop=True)
    val_df = pick(val_df, 30).reset_index(drop=True)
    log(f"QUICK: train {len(train_df)} / val {len(val_df)} đơn vị")

In [ ]:
# ===== 3. HÀM ĐÁNH GIÁ =====
def metrics_dict(y, pred):
    f1c = f1_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"macro_f1": f1_score(y, pred, labels=[0, 1, 2], average="macro", zero_division=0),
            "accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "qwk": cohen_kappa_score(y, pred, weights="quadratic"),
            "f1_ineffective": f1c[0], "f1_adequate": f1c[1], "f1_effective": f1c[2],
            "recall_ineffective": recall_score(y == 0, pred == 0, zero_division=0),
            "precision_ineffective": precision_score(y == 0, pred == 0, zero_division=0)}


def macro(y, p):
    return f1_score(y, p, labels=[0, 1, 2], average="macro", zero_division=0)


def paired_bootstrap(essays, y, base, new, n_boot=2000, seed=42):
    """Δ Macro-F1 (new − base), lấy mẫu lại theo bài."""
    codes = pd.Series(essays).astype("category").cat.codes.values
    members = [np.flatnonzero(codes == k) for k in range(codes.max() + 1)]
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(members), len(members))])
        d.append(macro(y[idx], new[idx]) - macro(y[idx], base[idx]))
    d = np.array(d)
    return {"delta": round(macro(y, new) - macro(y, base), 4), "ci_low": round(np.percentile(d, 2.5), 4),
            "ci_high": round(np.percentile(d, 97.5), 4), "p_one_sided_le0": round(float((d <= 0).mean()), 4),
            "n_units": len(y), "n_essays": len(members), "n_boot": n_boot}


def save_predictions(probs, name):
    out = val_df[["row_idx", "essay_id_comp", "discourse_type", "prompt_name"]].copy()
    out["true_label"] = val_df["label"].values
    out["pred_label"] = probs.argmax(1)
    for c in range(3):
        out[f"pred_proba_{c}"] = np.round(probs[:, c], 6)
    path = f"{OUT_DIR}/val_predictions_{name}.csv"
    out.to_csv(path, index=False)
    log(f"Lưu {path}")
    return out


def compare_with_reference(pred, name, subsets=None):
    """So với L0 / L1 tuần 11 trên CÙNG đơn vị validation, bootstrap theo bài."""
    ref1 = REF.set_index("row_idx").loc[val_df["row_idx"], "pred_label"].values
    ref0 = REF_L0.set_index("row_idx").loc[val_df["row_idx"], "pred_label"].values
    y, ess = val_df["label"].values, val_df["essay_id_comp"].values
    subsets = subsets or {"all": np.ones(len(y), bool)}
    rows = []
    for sname, m in subsets.items():
        if m.sum() == 0:
            continue
        for bname, base in [("L1 (tuần 11)", ref1), ("L0 (tuần 11)", ref0)]:
            r = paired_bootstrap(ess[m], y[m], base[m], pred[m], n_boot=200 if QUICK else 2000)
            rows.append({"model": name, "subset": sname, "baseline": bname,
                         "macro_f1_model": round(macro(y[m], pred[m]), 4),
                         "macro_f1_baseline": round(macro(y[m], base[m]), 4), **r})
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT_DIR}/compare_{name}_vs_L0_L1.csv", index=False)
    print(df.to_string(index=False))
    return df

In [ ]:
# ===== LOPO: 3 đề =====
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer,
                          TrainingArguments, set_seed)

FOLDS = {1: "A Cowboy Who Rode the Waves", 2: "Summer projects", 3: "Distance learning"}
EXPECTED = {1: 5158, 2: 9855, 3: 18526}
CFG = dict(model_name="microsoft/deberta-v3-base", max_length=512, eff_batch=32, lr=2e-5, epochs=3,
           warmup_ratio=0.1, seed=42)
if QUICK:
    CFG.update(epochs=1)
ALL = units  # LOPO dùng toàn bộ 144.289 đơn vị có nhãn
ALL["text"] = ALL["discourse_type"] + " [SEP] " + ALL["discourse_text"]


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device, logits.dtype))(
            logits.view(-1, 3), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); set_seed(seed)


def run_fold(fold):
    prompt = FOLDS[fold]
    pred_path = f"{OUT_DIR}/predictions_fold{fold}.csv"
    if os.path.exists(pred_path):
        log(f"fold {fold}: đã có kết quả, bỏ qua"); return
    tr = ALL[ALL["prompt_name"] != prompt].reset_index(drop=True)
    te = ALL[ALL["prompt_name"] == prompt].reset_index(drop=True)
    if QUICK:
        rng = np.random.RandomState(0)
        pick = lambda d, k: d[d.essay_id_comp.isin(rng.choice(d.essay_id_comp.unique(), k, replace=False))]
        tr, te = pick(tr, 60).reset_index(drop=True), pick(te, 20).reset_index(drop=True)
    else:
        assert len(te) == EXPECTED[fold], len(te)
    assert prompt not in set(tr["prompt_name"])
    log(f"fold {fold}: giấu '{prompt}' | train {len(tr):,} đơn vị / {tr.prompt_name.nunique()} đề | "
        f"test {len(te):,} đơn vị / {te.essay_id_comp.nunique():,} bài")
    seed = CFG["seed"] + fold
    seed_all(seed)
    tok = AutoTokenizer.from_pretrained(CFG["model_name"], use_fast=True)

    def to_hf(d):
        ds = Dataset.from_pandas(d[["text", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
        ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=CFG["max_length"]),
                    batched=True, batch_size=2000, remove_columns=["text"])
        return ds.map(lambda b: {"length": [len(x) for x in b["input_ids"]]}, batched=True)

    hf_tr, hf_te = to_hf(tr), to_hf(te)
    weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=tr["label"].values)
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG["model_name"], num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID).cuda()
    per_dev, accum = 16, 2          # L4 22 GB: 32 x 512 tràn, 16 x 512 đỉnh 12,4 GB (đã đo)
    args = TrainingArguments(
        output_dir=f"{WORK}/ckpt_lopo{fold}", learning_rate=CFG["lr"],
        per_device_train_batch_size=per_dev, per_device_eval_batch_size=128,
        gradient_accumulation_steps=accum, num_train_epochs=CFG["epochs"], warmup_ratio=CFG["warmup_ratio"],
        bf16=USE_BF16, fp16=not USE_BF16, tf32=True, group_by_length=True, length_column_name="length",
        dataloader_num_workers=4, eval_strategy="no", save_strategy="no",   # không chọn epoch trên đề bị giấu
        seed=seed, logging_steps=10 if QUICK else 200, report_to="none")
    trainer = WeightedLossTrainer(model=model, args=args, train_dataset=hf_tr,
                                  data_collator=DataCollatorWithPadding(tok), processing_class=tok,
                                  class_weights=torch.tensor(weights, dtype=torch.float))
    t0 = time.time()
    trainer.train()
    minutes = (time.time() - t0) / 60
    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    assert losses and np.isfinite(losses[-1]), losses[-5:]
    from torch.utils.data import SequentialSampler
    trainer._get_eval_sampler = lambda ds, *a, **k: SequentialSampler(ds)   # giữ đúng thứ tự (xem notebook 08)
    pred_out = trainer.predict(hf_te)
    assert (pred_out.label_ids == te["label"].values).all(), "Thứ tự dự đoán lệch"
    logits = pred_out.predictions
    logits = logits[0] if isinstance(logits, tuple) else logits
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), -1).numpy()
    y, pred = te["label"].values, probs.argmax(1)
    m = {k: round(float(v), 4) for k, v in metrics_dict(y, pred).items()}
    out = te[["row_idx", "essay_id_comp", "discourse_type", "prompt_name"]].copy()
    out["true_label"], out["pred_label"] = y, pred
    for c in range(3):
        out[f"pred_proba_{c}"] = np.round(probs[:, c], 6)
    out.to_csv(pred_path, index=False)
    json.dump({"fold": fold, "held_out_prompt": prompt, "n_train_units": len(tr), "n_test_units": len(te),
               "n_test_essays": int(te.essay_id_comp.nunique()), **m, "train_minutes": round(minutes, 1),
               "first_loss": losses[0], "last_loss": losses[-1], "seed": seed, "cfg": CFG, "gpu": GPU,
               "transformers": TV, "quick": QUICK},
              open(f"{OUT_DIR}/metrics_fold{fold}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    log(f"fold {fold}: train {minutes:.1f} phút | loss {losses[0]:.3f} -> {losses[-1]:.3f} | "
        f"Macro-F1 {m['macro_f1']} | Acc {m['accuracy']} | F1 Ineff {m['f1_ineffective']}")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
run_fold(1)

In [ ]:
run_fold(2)

In [ ]:
run_fold(3)

In [ ]:
rows = [json.load(open(f"{OUT_DIR}/metrics_fold{f}.json", encoding="utf-8")) for f in FOLDS]
res = pd.DataFrame(rows)[["fold", "held_out_prompt", "n_train_units", "n_test_units", "n_test_essays", "macro_f1",
                          "accuracy", "balanced_accuracy", "qwk", "f1_ineffective", "f1_adequate", "f1_effective",
                          "recall_ineffective", "train_minutes"]]
res.to_csv(f"{OUT_DIR}/lopo_deberta_rerun_results.csv", index=False)
print(res.to_string(index=False))
bundle = f"{WORK}/results_{KEY}" + ("_QUICK" if QUICK else "") + ".zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUT_DIR)):
        zf.write(f"{OUT_DIR}/{fn}", f"{os.path.basename(OUT_DIR)}/{fn}")
print("Đã đóng gói:", bundle, os.path.getsize(bundle), "bytes")